# Marketplace Listing Integrity Agent

A ReAct agent that decides whether a marketplace listing's image and description genuinely match — built with **LangGraph**, grounded by a **RAG** knowledge base of category-specific verification policy, with an **LLM as the brain** deciding which tools to call and in what order.

This is a direct successor to [product-image-description-alignment](https://github.com/ebiarian/product-image-description-alignment), which built a fixed OCR + VQA pipeline that runs the same five checks on every listing regardless of product category. That pipeline is reused here as a set of *tools* — but the fixed order and fixed feature set are gone. The agent decides, per listing, what actually needs checking.

### Why this needed to be agentic, not another fixed pipeline
The original pipeline's own roadmap named this gap directly: *"Category-aware feature sets — different product categories need different features."* A wrong color is a return for a T-shirt and irrelevant for a phone charger. A rule-based pipeline can't express that without a growing pile of if/else branches per category. An agent that retrieves category policy before deciding what to check can.

### Pipeline at a glance

| Stage | What happens | Model |
|---|---|---|
| 1. Detect category | short constrained VQA question, image-grounded — not from the (untrusted) description | **VQA** (Moondream2) |
| 2. Detect specific product | scoped to that category's known products | **VQA** (Moondream2) |
| 3. Self-consistency check | does the product belong to the detected category? Retry once if not, then fall back to trusting category alone | **LLM** (brain) |
| 4. Hard-stop vs. description | does the image-grounded product agree with what the description claims? Disagreement ends the check immediately | **LLM** (brain) |
| 5. Retrieve category policy | exact ID lookup if the category is known; semantic search as fallback | **Embedding model** (nomic-embed-text) |
| 6. Build checklist | union of the policy's critical features and any other concrete claim in the description | **LLM** (brain) |
| 7. Verify each feature | OCR first, VQA fallback, loops until enough evidence | **OCR** / **VQA** |
| 8. Final verdict | hard veto on any critical-feature mismatch | **LLM** (brain) |

### What this notebook does
1. Builds the RAG knowledge base of category verification policies (grocery, electronics, apparel) in Chroma
2. Demonstrates retrieval alone — both the exact-ID fast path and the semantic-search fallback — before wiring it into the agent
3. Defines the five tools the agent can call, including the two-question category/product detection sequence
4. Builds the ReAct graph (`agent` ↔ `tools`, looping until the model is done)
5. Runs the agent against real product listings — a correct one, and three with injected mismatches (wrong brand, wrong size, and a wrong product entirely) — printing the full reasoning trace for each

> **Requires:** a local [Ollama](https://ollama.com) server running `qwen2.5:14b` (reasoning) and `nomic-embed-text` (embeddings). See the project README for setup.

## Setup

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.config import OLLAMA_MODEL, EMBED_MODEL
from src.policy_corpus import CATEGORY_POLICIES
from src.retrieval import build_policy_corpus, retrieve_category_policy, get_vectorstore
from src.tools import ALL_TOOLS
from src.agent import build_graph, verify_listing
from src.listings import LISTINGS

print(f"Reasoning model : {OLLAMA_MODEL}")
print(f"Embedding model : {EMBED_MODEL}")
print(f"Tools available : {[t.name for t in ALL_TOOLS]}")

---
## Step 1: Building the Category Policy Knowledge Base (RAG)

Each category gets a short policy document — written the way an internal trust & safety wiki page would be, naming which features are **critical** (any mismatch is an automatic flag) versus **cosmetic** (a mismatch is only a soft warning), plus a short list of that category's common products (`common_products`) used later for VQA question generation. This is the knowledge the agent doesn't have baked into its prompt — it has to retrieve it.

In [ ]:
for policy in CATEGORY_POLICIES:
    print(f"=== {policy['category']} ===")
    if policy.get("common_products"):
        print(f"common_products: {policy['common_products']}")
    print(policy["text"])
    print()

In [ ]:
# Embeds every policy document into Chroma. Safe to re-run — it only
# re-embeds when the collection is empty or force_rebuild=True.
vectorstore = build_policy_corpus()
print(f"Collection size: {len(vectorstore.get()['ids'])} documents")

### Proving retrieval works both ways: exact lookup, and semantic fallback

`retrieve_category_policy` tries an **exact ID lookup** first (the category names ARE the document IDs — see `build_policy_corpus`), and only falls back to **embedding similarity search** when the input isn't an exact category name. The first two queries below hit the fast exact path; the rest have never been given as a literal category name, so they exercise the fallback — this is where RAG actually earns its keep.

In [ ]:
test_queries = [
    "grocery",                                  # exact category ID -> fast path, no embedding model involved
    "electronics",                               # exact category ID -> fast path
    "a pair of wireless bluetooth headphones",   # not an ID -> falls back to semantic search
    "a cotton crew-neck t-shirt",                # not an ID -> falls back to semantic search
]

for query in test_queries:
    result = retrieve_category_policy.invoke(query)
    retrieved_category = result.split("Category:")[1].split("(")[0].strip().rstrip(".")
    print(f"Query: '{query}'")
    print(f"  -> retrieved category: {retrieved_category}")
    print()

---
## Step 2: Tools the Agent Can Call

Five tools, each with a docstring written as an instruction to the model (this is what the LLM actually reads to decide when to call each one, and in what order):

| Tool | Role |
|---|---|
| `detect_category` | VQA — determines the product's category directly from the image |
| `detect_product` | VQA — determines the specific product, scoped to the detected category |
| `retrieve_category_policy` | RAG retrieval — what's critical vs. cosmetic for this listing's category |
| `read_label_text` | OCR (EasyOCR) — fast, reliable for anything printed on the label |
| `ask_vision_question` | Moondream2 VQA — one targeted feature question at a time, used as an OCR fallback |

In [ ]:
for t in ALL_TOOLS:
    print(f"### {t.name}")
    print(t.description.strip())
    print()

---
## Step 3: Building the ReAct Graph

Standard LangGraph ReAct shape: an `agent` node (the LLM bound to tools) and a `tools` node (`ToolNode`), wired with `tools_condition` so execution loops `agent -> tools -> agent` for as many steps as the model decides it needs, exiting to `END` only once it responds without requesting another tool call. A `recursion_limit` (see `src/config.py`) caps total steps as a safety net against a runaway loop.

In [ ]:
graph = build_graph()

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Diagram rendering unavailable, printing graph structure instead:")
    print(graph.get_graph().draw_mermaid())

---
## Step 4: Running the Agent — Correct Listing

The full reasoning trace is printed message by message: every tool call the agent makes (starting with `detect_category` and `detect_product`, never OCR/VQA feature checks first), every tool's response, and the model's final verdict.

In [ ]:
def print_trace(final_state):
    for m in final_state["messages"]:
        role = m.__class__.__name__.replace("Message", "")
        if role == "AI" and getattr(m, "tool_calls", None):
            for call in m.tool_calls:
                print(f"[{role} -> tool call] {call['name']}({call['args']})")
        elif role == "Tool":
            print(f"[Tool result] {m.content[:300]}")
        elif role == "AI":
            print(f"[{role} -> final answer]\n{m.content}")
        # SystemMessage / HumanMessage intentionally not printed — see the
        # listing dict above for the same information.
    print("\n" + "=" * 70 + "\n")

In [ ]:
listing = LISTINGS[0]
print(f"Listing: {listing['name']}")
print(f"Description: {listing['description_features']}\n")

result = verify_listing(graph, listing["image_url"], listing["description_features"])
print_trace(result)

---
## Step 5: Running the Agent — Injected Mismatches

The remaining listings, all against the *same real images* used above:
- **Wrong brand** (milk) and **wrong size** (butter) — critical-feature mismatches caught by the OCR/VQA verification loop (Stage 7)
- **Wrong product entirely** (the milk photo, described as orange juice) — this one should be caught much earlier, at the `detect_product` vs. description hard-stop (Stage 4), before any OCR/VQA feature checking happens at all

Watch the tool-call trace for that last one specifically — it should be visibly shorter than the others, since the agent has no reason to keep checking brand/size/variant once the product itself is already confirmed wrong.

In [ ]:
for listing in LISTINGS[1:]:
    print(f"Listing: {listing['name']}")
    print(f"Description: {listing['description_features']}\n")
    result = verify_listing(graph, listing["image_url"], listing["description_features"])
    print_trace(result)

---
## Summary

### What this agent does differently from the original pipeline
The original `product-image-description-alignment` pipeline runs OCR and all five Moondream2 questions on every listing, every time, regardless of category — a fixed, deterministic sequence. This agent instead:

- **Detects category and product from the image first, not from the description** — the description is exactly what's being verified, so nothing here trusts it until the image itself has spoken
- **Uses a two-question VQA hierarchy** (category, then product scoped to that category) rather than one open-ended or one flat-list question — meaningfully more reliable for a small local VQA model
- **Self-corrects on inconsistent evidence** — if the detected product doesn't belong to the detected category, the agent retries once before falling back to the more reliable category-level signal alone
- **Retrieves category policy before deciding what to check** — exact ID lookup on the fast path, RAG/semantic search as the fallback for anything that doesn't match a known category exactly
- **Builds a dynamic checklist** — the policy's critical features, plus any other concrete claim in the description even if the policy doesn't name it
- **Applies category-specific veto logic** — the same "wrong color" evidence would be treated very differently for a T-shirt versus a phone charger, because the retrieved policy — not the code — makes that call

### Key findings
- The exact-ID-first retrieval path means most listings never need the embedding model at all — RAG's semantic search only earns its keep on the fallback path, when the image-detected category doesn't cleanly resolve to a known ID
- A "wrong product entirely" mismatch resolves in far fewer tool calls than a fine-grained one (wrong brand/size) — the agent has no reason to check individual features once the product itself is already confirmed wrong
- The hard veto behaviour carries over from the original pipeline's design: any single critical-feature mismatch is enough to flag a listing, regardless of how many other features pass

### Limitations
- **Category corpus is small and synthetic** — three categories plus a default, written for this demo, not sourced from a real trust & safety policy document
- **`common_products` lists are hand-curated and narrow** — a genuinely novel product within a known category (e.g. a grocery item that isn't milk/coffee/butter/chicken/bread/juice) still gets forced into "something else" by `detect_product`, the same "not exhaustive at marketplace scale" limitation the original project already named for its own VQA prompts, just relocated here
- **Live testing is grocery-only** — the RAG corpus contains electronics and apparel policies and retrieval correctly discriminates between all three (Step 1), but the full agent has only been exercised end-to-end on real grocery images
- **Local Ollama reasoning** — `qwen2.5:14b` is capable but smaller and less reliable at multi-step tool orchestration than a frontier cloud model; a genuinely ambiguous listing may need more tool calls, or exhaust the one-time self-consistency retry without truly resolving
- **No human-in-the-loop yet** — every verdict is fully automatic; a production system would likely gate "Likely mismatch" verdicts on high-value categories behind human review, the way the sibling ridesharing project gates critical-severity decisions

### Potential future work
- Add real verified images for electronics and apparel to close the live-testing gap named above
- A human-in-the-loop gate (`interrupt()`) before a "Likely mismatch" verdict actually executes, for categories or sellers above a risk threshold
- Expand the policy corpus and `common_products` lists with real category taxonomy data instead of hand-written synthetic ones
- Memory across repeated checks on the same seller, to build an account-level risk signal instead of treating every listing independently